In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# Dataset adapter: pipeline output uses vi,ch while the model uses vi,cn internally.

EXPECTED_COLUMNS = ["vi", "ch"]

def load_poem_csv(path):
    frame = pd.read_csv(
        path,
        encoding="utf-8-sig",
        dtype=str,
        keep_default_na=False,
    )
    if list(frame.columns) != EXPECTED_COLUMNS:
        raise ValueError(
            f"{path}: expected columns {EXPECTED_COLUMNS}, found {list(frame.columns)}"
        )
    if frame.empty:
        raise ValueError(f"{path}: dataset is empty")

    blank_rows = frame[EXPECTED_COLUMNS].apply(lambda column: column.str.strip().eq("")).any(axis=1)
    if blank_rows.any():
        raise ValueError(f"{path}: found {int(blank_rows.sum())} rows with an empty vi or ch value")

    return (
        frame.rename(columns={"ch": "cn"})[["vi", "cn"]]
        .to_dict(orient="records")
    )

In [ ]:
# The train/validation/test CSV files are loaded below after their Kaggle paths are defined.

In [ ]:
# Vocabularies are rebuilt from poem.train.csv only to prevent test/validation leakage.

### Câu thơ dùng để smoke test

- **Phiên âm:** Độc nhiễu hư trai kính, / Thường trì tiểu phủ kha.
- **Chữ Hán:** 獨繞虛齋徑， / 常持小斧柯。
- **Nguồn:** Bài thơ "Ác thụ" – Đỗ Phủ – [**Phiêu bạc tây nam (760-770)**](https://www.thivien.net/%C4%90%E1%BB%97-Ph%E1%BB%A7/Phi%C3%AAu-b%E1%BA%A1c-t%C3%A2y-nam-760-770/group-ka14g6eudSWc8vsbQE81DQ)

In [ ]:
# Cell 1 - Imports + paths

import os

import torch
from torch.optim import AdamW
from model import (
    VietHanBertConfig,
    VietHanBertModel,
    VietHanTokenizer,
    build_vocabularies,
    character_bleu,
    create_data_loader,
    evaluate_loss,
    generate_dataset,
    generate_han,
    move_batch_to_device,
    save_checkpoint,
    save_model_bundle,
    save_vocabularies,
    tokenize_han,
    tokenize_vi,
)

# Change INPUT_DIR if the three generated CSV files are attached under another Kaggle dataset path.
INPUT_DIR = "/kaggle/input/datasets/tiennhat/dataset"
WORK_DIR = "/kaggle/working/viet_han_bert"

TRAIN_PATH = os.path.join(INPUT_DIR, "poem.train.csv")
VAL_PATH = os.path.join(INPUT_DIR, "poem.val.csv")
TEST_PATH = os.path.join(INPUT_DIR, "poem.test.csv")

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(os.path.join(WORK_DIR, "checkpoints"), exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Train:", TRAIN_PATH)
print("Validation:", VAL_PATH)
print("Test:", TEST_PATH)

In [ ]:
# Cell 2 - Load poem datasets

train_data = load_poem_csv(TRAIN_PATH)
val_data = load_poem_csv(VAL_PATH)
test_data = load_poem_csv(TEST_PATH)

print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))
print("Sample:", train_data[0])

In [ ]:
# Cell 3 - Tokenization smoke test

POEM_TEST_VI = "Độc nhiễu hư trai kính,\nThường trì tiểu phủ kha."
POEM_TEST_HAN = "獨繞虛齋徑，\n常持小斧柯。"

print("VI tokens:", tokenize_vi(POEM_TEST_VI))
print("Han tokens:", tokenize_han(POEM_TEST_HAN))

In [ ]:
# Cell 4 - Build Vietnamese and Han vocabularies from train only

vocab_vi, vocab_han = build_vocabularies(train_data)
VI_VOCAB_PATH, HAN_VOCAB_PATH = save_vocabularies(
    vocab_vi,
    vocab_han,
    os.path.join(WORK_DIR, "vocab"),
)

print("Vietnamese vocab:", len(vocab_vi))
print("Han vocab:", len(vocab_han))
print("Saved:", VI_VOCAB_PATH)
print("Saved:", HAN_VOCAB_PATH)

In [ ]:
# Cell 5 - Create tokenizer

tokenizer = VietHanTokenizer(VI_VOCAB_PATH, HAN_VOCAB_PATH)

print("VI vocab:", tokenizer.vi_vocab_size)
print("Han vocab:", tokenizer.han_vocab_size)

In [ ]:
# Cell 6 - Test tokenizer

print("VI tokens:", tokenizer.tokenize_vi(POEM_TEST_VI))
print("VI ids:", tokenizer.encode_vi(POEM_TEST_VI))

han_ids = tokenizer.encode_han(POEM_TEST_HAN)
print("Han tokens:", tokenizer.tokenize_han(POEM_TEST_HAN))
print("Han ids:", han_ids)
print("Han decoded:", tokenizer.decode_han(han_ids))

In [ ]:
# VietHanDataset is implemented in model/data.py.

In [ ]:
# VietHanCollator is implemented in model/data.py.

In [ ]:
# Cell 9 - Create datasets and loaders

MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128
BATCH_SIZE = 16

loader_options = {
    "batch_size": BATCH_SIZE,
    "max_source_length": MAX_SOURCE_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "num_workers": 2,
    "pin_memory": True,
}
train_loader = create_data_loader(train_data, tokenizer, shuffle=True, **loader_options)
val_loader = create_data_loader(val_data, tokenizer, **loader_options)
test_loader = create_data_loader(test_data, tokenizer, **loader_options)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
# Cell 10 - Inspect one batch

batch = next(iter(train_loader))

for key, value in batch.items():
    print(key, value.shape)
    print(value[0])

In [ ]:
# Cell 13 - Build model

config = VietHanBertConfig(
    vocab_size=tokenizer.vi_vocab_size,
    hidden_size=512,
    num_hidden_layers=6,
    num_attention_heads=8,
    intermediate_size=2048,
    max_position_embeddings=MAX_SOURCE_LENGTH,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
    han_vocab_size=tokenizer.han_vocab_size,
    decoder_layers=4,
    decoder_heads=8,
    decoder_ffn_dim=2048,
    max_target_position_embeddings=MAX_TARGET_LENGTH
)

model = VietHanBertModel(config).to(DEVICE)

num_params = sum(
    param.numel()
    for param in model.parameters()
)

print("Parameters:", f"{num_params:,}")

In [ ]:
# Cell 14 - Forward-pass smoke test

batch = move_batch_to_device(next(iter(train_loader)), DEVICE)
model.eval()
with torch.no_grad():
    outputs = model(**batch)

print("Loss:", outputs["loss"].item())
print("Logits:", outputs["logits"].shape)

In [ ]:
# Cell 15 - Overfit subset of 100 samples

small_loader = create_data_loader(
    train_data[:100],
    tokenizer,
    batch_size=8,
    max_source_length=MAX_SOURCE_LENGTH,
    max_target_length=MAX_TARGET_LENGTH,
    shuffle=True,
)
small_model = VietHanBertModel(config).to(DEVICE)
optimizer = AdamW(small_model.parameters(), lr=1e-4, weight_decay=0.0)
small_model.train()

for epoch in range(20):
    total_loss = 0.0
    for batch in small_loader:
        outputs = small_model(**move_batch_to_device(batch, DEVICE))
        loss = outputs["loss"]
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(small_model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch + 1:02d} | Loss: {total_loss / len(small_loader):.4f}")

In [ ]:
# Cell 16 - Training config

EPOCHS = 5
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
GRAD_ACCUM_STEPS = 2

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [ ]:
# evaluate_loss is implemented in model/training.py.

In [ ]:
# Cell 18 - Full training

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
model.train()

for epoch in range(EPOCHS):
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader):
        batch = move_batch_to_device(batch, DEVICE)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(**batch)
            loss = outputs["loss"]
            scaled_loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(scaled_loss).backward()
        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item()
        if step % 100 == 0:
            print(
                f"Epoch {epoch + 1} | Step {step}/{len(train_loader)} | "
                f"Loss {loss.item():.4f}"
            )

    train_loss = total_loss / len(train_loader)
    val_loss = evaluate_loss(model, val_loader, DEVICE)
    print(
        f"\nEpoch {epoch + 1} | Train Loss: {train_loss:.4f} | "
        f"Validation Loss: {val_loss:.4f}"
    )
    save_checkpoint(
        model,
        config,
        os.path.join(WORK_DIR, "checkpoints", f"epoch_{epoch + 1}.pt"),
    )

In [ ]:
# generate_han is implemented in model/generation.py.

In [ ]:
# Cell 20 - Inference test

model.eval()

prediction = generate_han(
    model,
    tokenizer,
    POEM_TEST_VI,
    DEVICE,
    max_length=MAX_TARGET_LENGTH,
)

print("VI      :", POEM_TEST_VI)
print("EXPECTED:", POEM_TEST_HAN)
print("PREDICT :", prediction)

In [ ]:
# Cell 21 - Test generation on test set

model.eval()

NUM_TEST_SAMPLES = 20

for i, item in enumerate(test_data[:NUM_TEST_SAMPLES]):
    source_text = item["vi"]
    target_text = item["cn"]

    prediction = generate_han(
        model,
        tokenizer,
        source_text,
        DEVICE,
        max_length=MAX_TARGET_LENGTH
    )

    print(f"[{i + 1}]")
    print("VI    :", source_text)
    print("TARGET:", target_text)
    print("PRED  :", prediction)
    print("-" * 80)

In [ ]:
# Cell 21 - Full test evaluation

sources, references, predictions = generate_dataset(
    model,
    tokenizer,
    test_data,
    DEVICE,
    max_source_length=MAX_SOURCE_LENGTH,
    max_length=MAX_TARGET_LENGTH,
    log_every=100,
)

print("Finished.")
print("Total samples:", len(predictions))

In [ ]:
# Cell 22 - Test metrics
# !pip install -q sacrebleu

char_bleu = character_bleu(predictions, references)
print("=" * 50)
print("CHARACTER-LEVEL BLEU")
print("=" * 50)
print(f"BLEU: {char_bleu.score:.2f}")
print("=" * 50)

In [ ]:
for i in range(20):
    print(f"[{i + 1}]")
    print("VI    :", sources[i])
    print("TARGET:", references[i])
    print("PRED  :", predictions[i])
    print("-" * 80)

In [ ]:
# Cell 23 - Save final model + tokenizer

FINAL_DIR = save_model_bundle(
    model,
    config,
    vocab_vi,
    vocab_han,
    os.path.join(WORK_DIR, "final"),
)
print("Saved to:", FINAL_DIR)